# Issue #4 — Conferência das mesorregiões (Eric)

**Objetivo:** validar o alvo `Mesorregiao` derivado de `Cidade`, verificar cobertura, variações de nomes, 15 municípios amostrados, contagens por posto e alterações necessárias.

**Referência oficial:** [Condepe/Fidem — Mesorregiões de Pernambuco (classificação histórica do IBGE)](https://www.cf.pe.gov.br/observatorio/mapas/regioes/mesorregioes), consultada em 23/11/2026; consultar também [IBGE — Divisão Regional do Brasil em Mesorregiões e Microrregiões, páginas 52–55](https://www.ibge.gov.br/biblioteca/visualizacao/monografias/GEBIS%20-%20RJ/DRB/Divisao%20regional_v01.pdf).

> Importante: as **mesorregiões históricas** são o alvo do projeto. O IBGE adotou **regiões geográficas intermediárias e imediatas** em 2017; não misturar as duas classificações.

> **Correções aplicadas** ao `src/meso_map.py`: Afrânio, Dormentes, Itacuruba, Terra Nova → São Francisco Pernambucano; Feira Nova e Machados → Agreste Pernambucano. A fonte já confirma as seis correções. Este notebook lê a versão corrigida do mapeamento.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Funciona tanto executado na pasta raiz quanto na pasta notebooks/.
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'data.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from data import carregar
from meso_map import meso_map

arquivo = next((ROOT / 'data' / 'raw').glob('*.csv'))
df = carregar(arquivo)
print(f'Registros: {len(df)} | Cidades (strings distintas): {df.Cidade.nunique()} | Códigos de posto: {df["Código"].nunique()}')
print('Linhas sem mesorregião:', df.Mesorregiao.isna().sum())
assert df.Mesorregiao.notna().all(), 'Há registros sem rótulo!'


Registros: 5104 | Cidades (strings distintas): 186 | Códigos de posto: 319
Linhas sem mesorregião: 0


## 1. Cobertura de cidades

Uma cidade com nome não encontrado no dicionário produziria um alvo nulo. Aqui também exibimos os nomes sem correspondência, caso a base mude.

In [2]:
nomes_dataset = set(df['Cidade'].unique())
sem_mapeamento = sorted(nomes_dataset - set(meso_map))
print('Nomes distintos do CSV:', len(nomes_dataset))
print('Nomes não mapeados:', sem_mapeamento)
assert not sem_mapeamento


Nomes distintos do CSV: 186
Nomes não mapeados: []


## 2. Variações de grafia

O dicionário conserva variantes de grafia que aparecem nas fontes. Nomes iguais após normalização precisam apontar para a mesma mesorregião.

In [3]:
pares_variantes = [
    ('Itaíba', 'Itaiba'), ('Calçado', 'Calçados'),
    ('Lagoa do Itaenga', 'Lagoa de Itaenga'),
    ('São Vicente Férrer', 'São Vicente Ferrer'),
    ('Camocim de São Felix', 'Camocim de São Félix'),
    ('Iguaraci', 'Iguaracy'), ('Jataúba', 'Jatauba'),
    ('Santa Maria do Cambucá', 'Santa Maria do Cambuca'),
]
variantes = pd.DataFrame([
    {'nome_1': a, 'nome_2': b, 'meso_1': meso_map.get(a), 'meso_2': meso_map.get(b),
     'linhas_nome_1': int((df.Cidade == a).sum()), 'linhas_nome_2': int((df.Cidade == b).sum()),
     'consistentes': meso_map.get(a) == meso_map.get(b)}
    for a, b in pares_variantes
])
display(variantes)
assert variantes.consistentes.all()


,nome_1,nome_2,meso_1,meso_2,linhas_nome_1,linhas_nome_2,consistentes
0,Itaíba,Itaiba,Agreste Pernambucano,Agreste Pernambucano,0,32,True
1,Calçado,Calçados,Agreste Pernambucano,Agreste Pernambucano,0,16,True
2,Lagoa do Itaenga,Lagoa de Itaenga,Mata Pernambucana,Mata Pernambucana,0,32,True
3,São Vicente Férrer,São Vicente Ferrer,Agreste Pernambucano,Agreste Pernambucano,16,0,True
4,Camocim de São Felix,Camocim de São Félix,Agreste Pernambucano,Agreste Pernambucano,16,0,True
5,Iguaraci,Iguaracy,Sertão Pernambucano,Sertão Pernambucano,16,0,True
6,Jataúba,Jatauba,Agreste Pernambucano,Agreste Pernambucano,16,16,True
7,Santa Maria do Cambucá,Santa Maria do Cambuca,Agreste Pernambucano,Agreste Pernambucano,0,16,True


## 3. Validação de 15 municípios

Amostra reprodutível de **15 nomes de cidades distintos**, selecionados com `random_state=42`. O campo `mesorregiao_referencia` foi transcrito da lista oficial Condepe/Fidem/IBGE, independentemente do dicionário do projeto; não é derivado de `meso_map`.

In [4]:
# Amostragem aleatória feita na base original (antes das correções)
amostra = sorted(df.Cidade.drop_duplicates().sample(15, random_state=42).tolist())
referencia_15 = {
    'Orobó': 'Agreste Pernambucano',
    'Surubim': 'Agreste Pernambucano',
    'Taquaritinga do Norte': 'Agreste Pernambucano',
    'Lagoa Grande': 'São Francisco Pernambucano',
    'Lagoa dos Gatos': 'Agreste Pernambucano',
    'Barreiros': 'Mata Pernambucana',
    'Tupanatinga': 'Agreste Pernambucano',
    'Calumbi': 'Sertão Pernambucano',
    'Paranatama': 'Agreste Pernambucano',
    'São João': 'Agreste Pernambucano',
    'Bonito': 'Agreste Pernambucano',
    'Ingazeira': 'Sertão Pernambucano',
    'São José do Egito': 'Sertão Pernambucano',
    'Panelas': 'Agreste Pernambucano',
    'Granito': 'Sertão Pernambucano',
}
assert set(amostra) == set(referencia_15), 'Amostra mudou; conferir fonte e referências novamente.'
validacao = pd.DataFrame({'Cidade':amostra})
validacao['mapeamento_projeto'] = validacao.Cidade.map(meso_map)
validacao['mesorregiao_referencia'] = validacao.Cidade.map(referencia_15)
validacao['correto'] = validacao.mapeamento_projeto.eq(validacao.mesorregiao_referencia)
display(validacao)
print('Acertos na amostra:', validacao.correto.sum(), 'de', len(validacao))
assert validacao.correto.all()


,Cidade,mapeamento_projeto,mesorregiao_referencia,correto
0,Barreiros,Mata Pernambucana,Mata Pernambucana,True
1,Bonito,Agreste Pernambucano,Agreste Pernambucano,True
2,Calumbi,Sertão Pernambucano,Sertão Pernambucano,True
3,Granito,Sertão Pernambucano,Sertão Pernambucano,True
4,Ingazeira,Sertão Pernambucano,Sertão Pernambucano,True
5,Lagoa Grande,São Francisco Pernambucano,São Francisco Pernambucano,True
6,Lagoa dos Gatos,Agreste Pernambucano,Agreste Pernambucano,True
7,Orobó,Agreste Pernambucano,Agreste Pernambucano,True
8,Panelas,Agreste Pernambucano,Agreste Pernambucano,True
9,Paranatama,Agreste Pernambucano,Agreste Pernambucano,True


Acertos na amostra: 15 de 15


## 4. Correções obrigatórias identificadas pela consulta à lista completa

O fato de a amostra ter 15 acertos **não significa** que o mapeamento completo estivesse correto. A comparação das listas das regiões detectou **6 municípios originalmente classificados em outra mesorregião**. A tabela abaixo documenta a correção, suas linhas e os rótulos atuais.

In [5]:
correcoes = {
    'Afrânio': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Dormentes': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Itacuruba': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Terra Nova': ('Sertão Pernambucano', 'São Francisco Pernambucano'),
    'Feira Nova': ('Mata Pernambucana', 'Agreste Pernambucano'),
    'Machados': ('Mata Pernambucana', 'Agreste Pernambucano'),
}
tabela_correcoes = pd.DataFrame([
    {'Cidade': cidade, 'antes': antes, 'depois': depois,
     'registros_afetados': int((df.Cidade == cidade).sum()),
     'postos_afetados': int(df.loc[df.Cidade == cidade, 'Código'].nunique()),
     'ajuste_aplicado': meso_map[cidade] == depois}
    for cidade, (antes, depois) in correcoes.items()
])
display(tabela_correcoes)
print('Registros corrigidos:', tabela_correcoes.registros_afetados.sum())
assert tabela_correcoes.ajuste_aplicado.all()


,Cidade,antes,depois,registros_afetados,postos_afetados,ajuste_aplicado
0,Afrânio,Sertão Pernambucano,São Francisco Pernambucano,32,2,True
1,Dormentes,Sertão Pernambucano,São Francisco Pernambucano,32,2,True
2,Itacuruba,Sertão Pernambucano,São Francisco Pernambucano,16,1,True
3,Terra Nova,Sertão Pernambucano,São Francisco Pernambucano,16,1,True
4,Feira Nova,Mata Pernambucana,Agreste Pernambucano,16,1,True
5,Machados,Mata Pernambucana,Agreste Pernambucano,16,1,True


Registros corrigidos: 128


## 5. Distribuição dos registros e postos

As contagens esperadas na issue refletem o dicionário **antes** da conferência. Depois da correção, elas devem mudar; o total de linhas permanece 5.104.

In [6]:
resumo = (df.groupby('Mesorregiao')
          .agg(registros=('Código', 'size'), postos=('Código', 'nunique'),
               cidades_strings=('Cidade', 'nunique'))
          .sort_values('registros', ascending=False))
display(resumo)
assert resumo.registros.sum() == 5104
assert resumo.postos.sum() == df['Código'].nunique()

postos_por_cidade = (df.groupby(['Mesorregiao','Cidade'])['Código']
                    .nunique().rename('quantidade_postos').reset_index()
                    .sort_values('quantidade_postos', ascending=False))
print('Municípios/nomes com mais de um posto:', int((postos_por_cidade.quantidade_postos > 1).sum()))
display(postos_por_cidade.query('quantidade_postos > 1').head(25))


,registros,postos,cidades_strings
Mesorregiao,,,
Agreste Pernambucano,1568,98,72
Sertão Pernambucano,1520,95,41
Mata Pernambucana,1024,64,43
Metropolitana do Recife,496,31,15
São Francisco Pernambucano,496,31,15


Municípios/nomes com mais de um posto: 64


,Mesorregiao,Cidade,quantidade_postos
131,Sertão Pernambucano,Araripina,14
162,Sertão Pernambucano,Sertânia,8
128,Metropolitana do Recife,Recife,6
64,Agreste Pernambucano,Taquaritinga do Norte,6
172,São Francisco Pernambucano,Belém de São Francisco,5
160,Sertão Pernambucano,Serra Talhada,5
19,Agreste Pernambucano,Caruaru,5
157,Sertão Pernambucano,Santa Cruz da Venerada,4
117,Metropolitana do Recife,Cabo,4
165,Sertão Pernambucano,São José do Egito,4


## 6. Conclusão e limitações

- **Cobertura:** 0/5.104 linhas sem rótulo.
- **Variantes:** os pares de grafia examinados são consistentes.
- **Amostra:** 15/15 nomes conferidos concordam com a classificação histórica oficial.
- **Correções:** 6 entradas no dicionário foram corrigidas, com impacto nas contagens de classe.
- **Postos:** o CSV contém 319 valores de `Código` distintos, em contraste com os 310 mencionados na documentação anterior; o número deve ser revisado pelo grupo.
- **Critério territorial:** mesorregiões históricas do IBGE, não as divisões geográficas vigentes a partir de 2017.
- **Limite:** a validação da amostra e a comparação das listas identificam problemas de atribuição; **não** auditam se o nome do posto da APAC está correto na origem. O dicionário ainda contém grafias alternativas e nomes históricos que devem permanecer documentados.

**Aviso ao grupo:** como o alvo `y` mudou em algumas linhas, recomenda-se regenerar os resultados dos 360 experimentos e as métricas do modelo com o dicionário corrigido. Não divulgar resultados antigos como se fossem da base revisada.

---
# Issue #13 — Valores incorretos, duplicatas e outliers (Eric)

Diagnóstico do CSV **sem alterar o dataset original**. As métricas do KNN usam a base v1 (`≤6` meses ausentes, imputação por mediana, StandardScaler, `k=5`, pesos por distância, Manhattan), `StratifiedGroupKFold` por posto (5 folds, seed 42). O mapeamento usado é o corrigido na Issue #4.

In [7]:
from data import MESES
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# Os códigos acima já definiram ROOT, df e MESES; relê-se o arquivo bruto para checagens lexicais.
bruto = pd.read_csv(next((ROOT/'data'/'raw').glob('*.csv')))
print('Dimensão:', bruto.shape)

Dimensão: (5104, 16)


## 7. Valores inválidos e extremos

`-` representa ausência no arquivo, **não** precipitação negativa; verificar explicitamente todas as demais strings antes da conversão numérica. Picos elevados são suspeitos para inspeção, não automaticamente erros físicos.

In [8]:
valores_brutos = bruto[MESES].astype(str).apply(lambda c: c.str.strip())
faltantes = valores_brutos.eq('-') | valores_brutos.eq('') | valores_brutos.eq('nan')
convertido = valores_brutos.apply(lambda c: pd.to_numeric(c.str.replace(',', '.', regex=False), errors='coerce'))
nao_numericos_inesperados = (~faltantes) & convertido.isna()
negativos = convertido.lt(0)
print('Tokens não numéricos inesperados:', int(nao_numericos_inesperados.sum().sum()))
print('Valores negativos:', int(negativos.sum().sum()))
print('Meses ausentes:', int(faltantes.sum().sum()))
print('Extremos mensais (mm):')
display(pd.DataFrame({'mínimo': convertido.min(), 'máximo': convertido.max(),
                      'acima_600': convertido.gt(600).sum()}).round(1))
picos = df.loc[df[MESES].gt(600).any(axis=1), ['Código','Posto','Ano','Mesorregiao']+MESES]
print('Registros com ao menos um valor > 600 mm:', len(picos))
display(picos.head(12))

Tokens não numéricos inesperados: 0
Valores negativos: 0
Meses ausentes: 25309
Extremos mensais (mm):


,mínimo,máximo,acima_600
Janeiro,0.1,401.0,0
Fevereiro,0.1,478.2,0
Março,0.1,625.0,1
Abril,0.2,758.3,8
Maio,0.2,891.5,37
Junho,0.2,1036.2,29
Julho,0.2,782.0,9
Agosto,0.1,546.1,0
Setembro,0.1,359.6,0
Outubro,0.1,317.3,0


Registros com ao menos um valor > 600 mm: 75


,Código,Posto,Ano,Mesorregiao,Janeiro,Fevereiro,Março,Abril,Maio,Junho,Julho,Agosto,Setembro,Outubro,Novembro,Dezembro
34,106,Barreiros,2010,Mata Pernambucana,163.3,110.2,139.4,173.9,106.2,610.1,4.3,NaN,NaN,NaN,NaN,NaN
35,314,Barreiros - PCD,2010,Mata Pernambucana,158.6,114.2,119.1,178.3,105.0,613.5,220.1,175.4,113.0,30.2,7.3,25.1
63,490,Cabo (Barragem de Gurjaú),2010,Metropolitana do Recife,267.5,277.6,230.9,433.3,316.7,1036.2,191.6,151.6,87.1,10.5,17.9,43.0
64,491,Cabo (Barragem de Suape),2010,Metropolitana do Recife,120.5,50.1,78.0,265.7,126.2,642.7,173.4,133.3,57.2,77.2,13.5,10.2
75,53,Canhotinho,2010,Agreste Pernambucano,149.5,65.1,152.0,141.3,58.7,650.9,151.8,28.8,33.6,16.0,NaN,51.2
89,284,Catende (Usina Catende),2010,Mata Pernambucana,94.0,67.5,55.0,153.5,38.0,611.5,156.2,105.0,126.5,6.9,2.5,20.5
176,392,Maraial (Usina São Luis),2010,Mata Pernambucana,134.0,39.5,44.4,134.5,77.5,664.0,121.8,88.8,165.1,81.0,7.3,22.9
183,209,Olinda (Alto da Bondade),2010,Metropolitana do Recife,75.9,36.6,93.9,237.7,127.1,600.2,178.2,136.5,66.9,55.6,13.0,87.1
190,25,Palmares,2010,Mata Pernambucana,119.7,93.5,65.8,172.0,53.7,659.3,160.5,295.0,73.1,15.1,NaN,17.5
223,543,Ribeirão,2010,Mata Pernambucana,265.8,74.1,131.2,236.4,67.4,639.0,142.0,118.0,63.5,25.8,10.9,12.0


## 8. Duplicatas e postos diferentes na mesma cidade

Duplicação de `Código` + `Ano` pode indicar repetição de uma observação; cidades com vários códigos são postos diferentes e **não** devem ser removidos automaticamente. Correlação entre séries anuais de postos da mesma cidade exige anos coincidentes e meses medidos em ambos.

In [9]:
print('Linhas inteiramente duplicadas:', int(bruto.duplicated().sum()))
print('Pares Código/Ano duplicados:', int(bruto.duplicated(['Código','Ano']).sum()))
codigos_cidade = df.groupby('Cidade')['Código'].nunique().sort_values(ascending=False)
display(codigos_cidade[codigos_cidade > 1].rename('n_postos').head(25).to_frame())
print('Cidades com múltiplos códigos:', int((codigos_cidade>1).sum()))
# Agregados anuais com pelo menos 1 mês válido; análise exploratória apenas.
anual = df.assign(total_anual=df[MESES].sum(axis=1, min_count=1))
pares = []
for cidade, sub in anual.groupby('Cidade'):
    tab = sub.pivot_table(index='Ano', columns='Código', values='total_anual', aggfunc='mean')
    ids = list(tab.columns)
    for i, a in enumerate(ids):
        for b in ids[i+1:]:
            comum = tab[[a,b]].dropna()
            if len(comum) >= 5 and comum[a].nunique()>1 and comum[b].nunique()>1:
                pares.append({'cidade':cidade,'posto_a':a,'posto_b':b,'anos_comuns':len(comum),
                              'corr_total_anual':comum[a].corr(comum[b])})
correlacoes_postos = pd.DataFrame(pares)
print('Pares de postos com >=5 anos comparáveis:',len(correlacoes_postos))
if len(correlacoes_postos):
    display(correlacoes_postos.sort_values('corr_total_anual',ascending=False).head(10).round(3))
    display(correlacoes_postos['corr_total_anual'].describe().round(3).to_frame())

Linhas inteiramente duplicadas: 0
Pares Código/Ano duplicados: 0


,n_postos
Cidade,
Araripina,14
Sertânia,8
Recife,6
Taquaritinga do Norte,6
Caruaru,5
Belém de São Francisco,5
Serra Talhada,5
Cabo,4
Olinda,4


Cidades com múltiplos códigos: 64


Pares de postos com >=5 anos comparáveis: 70


,cidade,posto_a,posto_b,anos_comuns,corr_total_anual
39,Olinda,209,551,7,0.995
7,Belém de São Francisco,9,257,6,0.988
37,Olinda,199,209,7,0.978
40,Ouricuri,175,373,8,0.967
38,Olinda,199,551,12,0.958
50,Santa Maria da Boa Vista,6,187,12,0.946
4,Belo Jardim,20,374,16,0.943
27,Floresta,32,178,16,0.940
49,Salgueiro,8,348,16,0.934
9,Belém de São Francisco,34,257,6,0.918


,corr_total_anual
count,70.000
mean,0.566
std,0.346
min,-0.468
25%,0.304
50%,0.648
75%,0.878
max,0.995


## 9. Outliers pelo IQR (1,5 × intervalo interquartil)

Os limites são calculados **por mês e mesorregião** apenas sobre valores observados. Este é um diagnóstico descritivo da base, não uma etapa de seleção do teste. Valores muito altos podem ser chuvas extremas legítimas; o método IQR não identifica erros por si só.

In [10]:
long = df.melt(id_vars=['Mesorregiao'], value_vars=MESES,
               var_name='Mês',value_name='Chuva_mm').dropna()
estat = (long.groupby(['Mesorregiao','Mês'])['Chuva_mm']
            .agg(q1=lambda x:x.quantile(.25), q3=lambda x:x.quantile(.75), n='size').reset_index())
estat['iqr'] = estat.q3 - estat.q1
estat['limite_inferior'] = estat.q1 - 1.5*estat.iqr
estat['limite_superior'] = estat.q3 + 1.5*estat.iqr
marcado = long.merge(estat,on=['Mesorregiao','Mês'],how='left')
marcado['outlier'] = (marcado.Chuva_mm < marcado.limite_inferior) | (marcado.Chuva_mm > marcado.limite_superior)
tabela_iqr = (marcado.groupby(['Mesorregiao','Mês'],as_index=False)
             .agg(n_observados=('Chuva_mm','size'),n_outliers=('outlier','sum')))
tabela_iqr['percentual'] = (100*tabela_iqr.n_outliers/tabela_iqr.n_observados).round(2)
display(tabela_iqr.pivot(index='Mesorregiao',columns='Mês',values='n_outliers').reindex(columns=MESES))
display(tabela_iqr.groupby('Mesorregiao')[['n_observados','n_outliers']].sum()
        .assign(percentual=lambda t:(100*t.n_outliers/t.n_observados).round(2)))
print('Total de células assinaladas como outlier:',int(marcado.outlier.sum()))

Mês,Janeiro,Fevereiro,Março,Abril,Maio,Junho,Julho,Agosto,Setembro,Outubro,Novembro,Dezembro
Mesorregiao,,,,,,,,,,,,
Agreste Pernambucano,30,32,37,25,51,30,52,37,65,79,91,33
Mata Pernambucana,7,26,20,23,24,25,19,15,39,45,37,18
Metropolitana do Recife,0,3,9,15,5,1,0,8,31,22,7,12
Sertão Pernambucano,11,23,19,17,30,34,30,24,22,27,21,26
São Francisco Pernambucano,23,7,16,13,19,27,20,23,14,10,11,2


,n_observados,n_outliers,percentual
Mesorregiao,,,
Agreste Pernambucano,12435,562,4.52
Mata Pernambucana,8494,298,3.51
Metropolitana do Recife,4355,113,2.59
Sertão Pernambucano,7494,284,3.79
São Francisco Pernambucano,3161,185,5.85


Total de células assinaladas como outlier: 1442


## 10. Experimento: base v1 × teto fixo de 600 mm

Comparamos *sem teto* versus *clip* fixo de `[0, 600]`, sem descartar nenhuma linha do conjunto de teste. A função de transformação de features no `Pipeline` é aplicada **igualmente em treino e previsão**, como exige o uso coerente do modelo; o teto é predefinido, não estimado a partir do teste. Imputação e normalização aprendem somente no treino de cada fold. Não aplicamos filtragem de registros baseada em outliers para evitar excluir observações do teste. Os números aqui podem diferir dos experimentos históricos porque os rótulos foram corrigidos na Issue #4.

In [11]:
dados = df.dropna(subset=['Mesorregiao'])
X = dados[MESES].copy(); y = dados['Mesorregiao'].copy(); grupos = dados['Código'].copy()
mascara = X.isna().sum(axis=1) <= 6
Xf, yf, gf = X.loc[mascara], y.loc[mascara], grupos.loc[mascara]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
def construir_pipeline(com_teto=False):
    etapas = [('imputer', SimpleImputer(strategy='median'))]
    if com_teto:
        etapas.append(('clip_600',FunctionTransformer(np.clip,kw_args={'a_min':0,'a_max':600})))
    etapas.extend([('scaler', StandardScaler()),
                   ('knn',KNeighborsClassifier(n_neighbors=5,weights='distance',metric='manhattan'))])
    return Pipeline(etapas)
linhas = []
for rotulo, teto in [('base v1',False),('teto 600 mm',True)]:
    pipe = construir_pipeline(teto)
    r = cross_validate(pipe,Xf,yf,groups=gf,cv=cv,n_jobs=1,
                       scoring=['accuracy','balanced_accuracy','f1_macro'])
    pred = cross_val_predict(pipe,Xf,yf,groups=gf,cv=cv,n_jobs=1)
    rel = classification_report(yf,pred,output_dict=True,zero_division=0)
    sf = rel['São Francisco Pernambucano']
    linhas.append({'config':rotulo,'n_instancias':len(Xf),
                  'acc':round(float(r['test_accuracy'].mean()),3),
                  'bal_acc':round(float(r['test_balanced_accuracy'].mean()),3),
                  'f1_macro':round(float(r['test_f1_macro'].mean()),3),
                  'f1_std':round(float(r['test_f1_macro'].std()),3),
                  'f1_SF':round(sf['f1-score'],3),'recall_SF':round(sf['recall'],3)})
resultados_limpeza = pd.DataFrame(linhas)
display(resultados_limpeza)
diferenca = resultados_limpeza.loc[1,'f1_macro'] - resultados_limpeza.loc[0,'f1_macro']
print(f'Diferença F1 macro (teto - base): {diferenca:+.3f}')
print('Recomendação automática:', 'não usar teto de 600 mm' if diferenca <= 0 else 'considerar teto de 600 mm, validando estabilidade e plausibilidade')

,config,n_instancias,acc,bal_acc,f1_macro,f1_std,f1_SF,recall_SF
0,base v1,3337,0.744,0.717,0.723,0.032,0.559,0.497
1,teto 600 mm,3337,0.742,0.714,0.720,0.031,0.553,0.494


Diferença F1 macro (teto - base): -0.003
Recomendação automática: não usar teto de 600 mm


## 11. Interpretação e recomendação

- `-` é tratado como ausência, não como número inválido. Não há exclusão adicional de linhas por IQR.
- Correlação elevada entre postos não significa duplicata; seus códigos e suas observações são distintos.
- Um valor acima de 600 mm **não é prova de erro** sem metadados meteorológicos.
- O resultado do experimento deve orientar a recomendação para a Issue #12 (Randell), sempre conferindo `f1_macro`, `f1_SF` e estabilidade entre folds.
- **Limitação:** a análise de correlação usa totais anuais de meses disponíveis; diferenças de completude podem influenciar a correlação.